In [ ]:
CLasswork

In [8]:
from transformers import AutoTokenizer, AutoModel
import tensorflow as tf
import torch # Import torch for PyTorch operations

# ==========================================
# 1. LOAD PRE-TRAINED GOOGLE BERT
# ==========================================
print("Downloading BERT weights... (This might take a few seconds)\n")

# Load the Tokenizer and the Model directly from Hugging Face
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# Load the model for PyTorch (default if from_tf=False or omitted and PyTorch is installed)
bert_model = AutoModel.from_pretrained("bert-base-uncased")

# The sentence we want to analyze
text = "Deep learning is incredibly powerful."

# ==========================================
# 2. THE CHALLENGE: EXTRACT THE CONTEXT
# ==========================================

# Step 1: Tokenize the text for PyTorch
# YOUR CODE HERE:
inputs = tokenizer(text, return_tensors="pt")

print("1. Tokenized Input IDs:", inputs["input_ids"])

# Step 2: Pass the inputs through the massive BERT model
# YOUR CODE HERE:
# Unpack the dictionary inputs using ** to pass individual tensors to the model
outputs = bert_model(**inputs)

# The output has a property called .last_hidden_state
# Shape is: (batch_size, sequence_length, hidden_dimension)
last_hidden_state = outputs.last_hidden_state

# Step 3: Extract ONLY the [CLS] token (The 0th position of the sequence)
# YOUR CODE HERE:
cls_token = last_hidden_state[:, 0, :]


print(f"\n2. Full Output Shape: {last_hidden_state.shape}")
print(f"3. [CLS] Token Shape: {cls_token.shape}")
# Ensure the tensor is on CPU before converting to numpy
print(f"\nFirst 10 numbers of our [CLS] Context Vector:\n{cls_token[0, :10].detach().cpu().numpy()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1. Tokenized Input IDs: tensor([[  101,  2784,  4083,  2003, 11757,  3928,  1012,   102]])

2. Full Output Shape: torch.Size([1, 8, 768])
3. [CLS] Token Shape: torch.Size([1, 768])

First 10 numbers of our [CLS] Context Vector:
[ 0.19359827  0.04280461 -0.24664204  0.19991781 -0.46639106 -0.3159976
  0.39997226  0.49262115  0.34630617 -0.59278417]


In [ ]:
BERT USAGE

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print("Loading model and tokenizer...\n")

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

texts = [
    "The new Batman movie had incredible action scenes.",
    "The plot was boring and the acting was completely flat."
]

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

probabilities = torch.nn.functional.softmax(logits, dim=-1);

predicted_class_ids = torch.argmax(probabilities, dim=-1).tolist()

labels = ["NEGATIVE", "POSITIVE"]

print("--- RESULTS ---")
for i, text in enumerate(texts):
    predicted_label = labels[predicted_class_ids[i]]
    confidence = probabilities[i][predicted_class_ids[i]].item()

    print(f"Text: '{text}'")
    print(f"Prediction: {predicted_label} (Confidence: {confidence:.2%})\n")

Loading model and tokenizer...



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--- RESULTS ---
Text: 'The new Batman movie had incredible action scenes.'
Prediction: POSITIVE (Confidence: 99.99%)

Text: 'The plot was boring and the acting was completely flat.'
Prediction: NEGATIVE (Confidence: 99.98%)

